# Wikipedia Event Stream Processing

This notebook monitors real-time Wikipedia events for selected entities from the IMDB dataset.

**Team Members:**
- Samuel Giorno
- Maxime Boiral
- Lou Bruneau

## Overview

We will:
1. Select 5 entities from our IMDB analysis
2. Monitor their Wikipedia pages for edit events
3. Track metrics (edit count, user activity, bytes changed)
4. Generate alerts for anomalous activity
5. Store data in SQLite database

## Entities Selected

Based on our IMDB analysis, we're monitoring:
1. **Christopher Nolan** - Acclaimed director
2. **The Shawshank Redemption** - Highly rated movie
3. **Quentin Tarantino** - Influential filmmaker
4. **The Godfather** - Classic film
5. **Steven Spielberg** - Legendary director

## Metrics Tracked

For each entity:
- **Edit count**: Total number of edits
- **Unique users**: Number of different editors
- **Anonymous edits**: Edits by non-registered users
- **Bot edits**: Automated edits
- **Bytes changed**: Total content changes
- **Edit frequency**: Edits per hour

## Alert Conditions

Alerts are generated when:
1. **High frequency**: >5 edits per hour for an entity
2. **Anonymous edits**: Any edit by non-registered user
3. **Rapid changes**: Unusual editing patterns

Alerts are stored separately in `outputs/alerts.json` for review.

## Setup and Imports

In [1]:
import sys
!{sys.executable} -m pip install sseclient


In [2]:
import sqlite3
import json
import pandas as pd
from datetime import datetime
import time

# Add src directory to path
sys.path.append('../src')
from stream_processor import WikipediaStreamProcessor

print("Libraries imported successfully!")
print(f"Current time: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

Libraries imported successfully!
Current time: 2025-12-01 22:31:08


## Define Entities to Monitor

In [3]:
# we select 5 entities with active Wikipedia pages
entities_to_monitor = [
    "Christopher Nolan",        
    "The Shawshank Redemption", 
    "Quentin Tarantino",        
    "The Godfather",            
    "Steven Spielberg"          
]

print("Entities selected for monitoring:")
for i, entity in enumerate(entities_to_monitor, 1):
    print(f"{i}. {entity}")

print(f"\nTotal: {len(entities_to_monitor)} entities")

Entities selected for monitoring:
1. Christopher Nolan
2. The Shawshank Redemption
3. Quentin Tarantino
4. The Godfather
5. Steven Spielberg

Total: 5 entities


## Initialize Stream Processor

In [4]:
# Create stream processor instance
processor = WikipediaStreamProcessor(
    entities=entities_to_monitor,
    db_path='../outputs/metrics.db',
    alert_path='../outputs/alerts.json'
)

print("\nStream processor initialized!")
print(f"Database: {processor.db_path}")
print(f"Alerts: {processor.alert_path}")
print(f"Alert threshold: {processor.alert_threshold} edits per hour")

✓ Database initialized: ../outputs/metrics.db
Stream processor initialized for 5 entities
Entities: Christopher Nolan, The Shawshank Redemption, Quentin Tarantino, The Godfather, Steven Spielberg

Stream processor initialized!
Database: ../outputs/metrics.db
Alerts: ../outputs/alerts.json
Alert threshold: 5 edits per hour


## Start Monitoring

### Stream Monitoring Duration

The stream processor will run for **5 minutes** by default to demonstrate functionality.

To run longer, modify `duration_seconds` parameter:
- `duration_seconds = 300` → 5 minutes (demo mode)
- `duration_seconds = 600` → 10 minutes
- `duration_seconds = None` → unlimited (stop manually with Kernel → Interrupt)

In [5]:
# For testing: monitor for 5 minutes (300 seconds)
# For production: use duration_seconds=None for infinite monitoring

try:
    processor.start_monitoring(duration_seconds=300)  # 5 minutes for demo
    # processor.start_monitoring(duration_seconds=None)  # Uncomment for infinite monitoring
except KeyboardInterrupt:
    print("\nMonitoring stopped by user")
except Exception as e:
    print(f"\nError: {e}")


Starting Wikipedia Event Stream Monitoring
Monitoring entities: Christopher Nolan, The Shawshank Redemption, Quentin Tarantino, The Godfather, Steven Spielberg
Database: ../outputs/metrics.db
Alerts: ../outputs/alerts.json
Duration: 300 seconds

Connecting to Wikimedia EventStreams...

Error during monitoring: Failed to parse: <Response [403]>

Saving final metrics...

METRICS SUMMARY



✓ Metrics saved to: ../outputs/metrics.db
✓ Alerts saved to: ../outputs/alerts.json
✓ Total alerts generated: 0


## View Collected Metrics

In [6]:
# Connect to database and view metrics
conn = sqlite3.connect('../outputs/metrics.db')

# Load entity metrics
df_metrics = pd.read_sql_query("SELECT * FROM entity_metrics ORDER BY timestamp DESC", conn)

print("Entity Metrics Summary:")
print("=" * 80)
display(df_metrics)

# Latest metrics per entity
print("\nLatest Metrics by Entity:")
print("=" * 80)
latest_metrics = df_metrics.groupby('entity_name').first().reset_index()
display(latest_metrics[['entity_name', 'edit_count', 'unique_users', 'anonymous_edits', 'bot_edits', 'bytes_changed']])

conn.close()

Entity Metrics Summary:


,id,entity_name,timestamp,edit_count,bytes_changed,unique_users,anonymous_edits,bot_edits



Latest Metrics by Entity:


,entity_name,edit_count,unique_users,anonymous_edits,bot_edits,bytes_changed


## View Edit Events

In [7]:
# View individual edit events
conn = sqlite3.connect('../outputs/metrics.db')

df_events = pd.read_sql_query("""
    SELECT * FROM edit_events 
    ORDER BY timestamp DESC 
    LIMIT 50
""", conn)

print("Recent Edit Events:")
print("=" * 80)
display(df_events)

# Event statistics by entity
print("\nEdit Events by Entity:")
print("=" * 80)
event_stats = df_events.groupby('entity_name').agg({
    'id': 'count',
    'is_anonymous': 'sum',
    'is_bot': 'sum',
    'bytes_changed': 'sum'
}).rename(columns={'id': 'total_events'})
display(event_stats)

conn.close()

Recent Edit Events:


,id,entity_name,timestamp,user,is_bot,is_anonymous,bytes_changed,comment



Edit Events by Entity:


,total_events,is_anonymous,is_bot,bytes_changed
entity_name,,,,


## View Alerts

In [8]:
# Load and display alerts
try:
    alerts = []
    with open('../outputs/alerts.json', 'r') as f:
        for line in f:
            alerts.append(json.loads(line))
    
    print(f"Total Alerts Generated: {len(alerts)}")
    print("=" * 80)
    
    if alerts:
        # Convert to DataFrame for better viewing
        df_alerts = pd.DataFrame(alerts)
        display(df_alerts)
        
        # Alert summary
        print("\nAlert Summary:")
        print("=" * 80)
        print(df_alerts.groupby(['entity', 'reason']).size().reset_index(name='count'))
    else:
        print("No alerts generated yet.")
        
except FileNotFoundError:
    print("Alert file not found. No alerts generated yet.")
except Exception as e:
    print(f"Error loading alerts: {e}")

Alert file not found. No alerts generated yet.


## Visualizations (Optional)

In [9]:
import matplotlib.pyplot as plt
import seaborn as sns

# Set style
sns.set_style('whitegrid')
plt.figure(figsize=(15, 10))

# Connect to database
conn = sqlite3.connect('../outputs/metrics.db')

# Load data
df_metrics = pd.read_sql_query("SELECT * FROM entity_metrics ORDER BY timestamp", conn)

if len(df_metrics) > 0:
    # Plot 1: Edit counts by entity
    plt.subplot(2, 2, 1)
    latest = df_metrics.groupby('entity_name')['edit_count'].last()
    latest.plot(kind='bar', color='steelblue')
    plt.title('Total Edits by Entity')
    plt.xlabel('Entity')
    plt.ylabel('Edit Count')
    plt.xticks(rotation=45, ha='right')
    
    # Plot 2: Unique users by entity
    plt.subplot(2, 2, 2)
    latest_users = df_metrics.groupby('entity_name')['unique_users'].last()
    latest_users.plot(kind='bar', color='coral')
    plt.title('Unique Users by Entity')
    plt.xlabel('Entity')
    plt.ylabel('Unique Users')
    plt.xticks(rotation=45, ha='right')
    
    # Plot 3: Anonymous vs Bot edits
    plt.subplot(2, 2, 3)
    edit_types = df_metrics.groupby('entity_name')[['anonymous_edits', 'bot_edits']].last()
    edit_types.plot(kind='bar', stacked=True)
    plt.title('Anonymous vs Bot Edits')
    plt.xlabel('Entity')
    plt.ylabel('Edit Count')
    plt.legend(['Anonymous', 'Bot'])
    plt.xticks(rotation=45, ha='right')
    
    # Plot 4: Edit count over time (if we have time series data)
    plt.subplot(2, 2, 4)
    for entity in df_metrics['entity_name'].unique():
        entity_data = df_metrics[df_metrics['entity_name'] == entity]
        if len(entity_data) > 1:
            plt.plot(pd.to_datetime(entity_data['timestamp']), 
                    entity_data['edit_count'], 
                    marker='o', 
                    label=entity)
    plt.title('Edit Count Over Time')
    plt.xlabel('Time')
    plt.ylabel('Cumulative Edits')
    plt.legend()
    plt.xticks(rotation=45, ha='right')
    
    plt.tight_layout()
    plt.show()
else:
    print("No metrics data available for visualization yet.")

conn.close()

No metrics data available for visualization yet.


<Figure size 1500x1000 with 0 Axes>

## Stream Processing Summary

### Architecture

The stream processing system consists of:

1. **Data Source**: Wikimedia EventStreams (Server-Sent Events)
   - URL: https://stream.wikimedia.org/v2/stream/recentchange
   - Real-time stream of all Wikipedia edits across all languages

2. **Event Filtering**: 
   - Monitors only edit events (not page creations, deletions, etc.)
   - Filters for our 5 selected entities
   - Matches entity names against page titles

3. **Metrics Collection**:
   - **Edit count**: Total edits per entity
   - **User tracking**: Unique editors and their edit counts
   - **Edit classification**: Anonymous, bot, or registered user
   - **Content changes**: Bytes added/removed
   - **Temporal data**: Edit timestamps and frequency

4. **Storage**:
   - **SQLite Database** (`metrics.db`):
     - `entity_metrics` table: Aggregated metrics snapshots
     - `edit_events` table: Individual edit records
   - **JSON File** (`alerts.json`):
     - Line-delimited JSON for alerts
     - Easy to parse and analyze separately

5. **Alerting System**:
   - **High frequency alert**: Triggers when >5 edits/hour detected
   - **Anonymous edit alert**: Flags all edits by unregistered users
   - Alerts stored separately for easy review and action

### Data Structure

**entity_metrics table:**
```
id | entity_name | timestamp | edit_count | bytes_changed | unique_users | anonymous_edits | bot_edits
```

**edit_events table:**
```
id | entity_name | timestamp | user | is_bot | is_anonymous | bytes_changed | comment
```

**alerts.json structure:**
```json
{
  "timestamp": "2024-11-27T10:30:00",
  "entity": "Christopher Nolan",
  "reason": "High edit frequency detected",
  "data": {
    "edits_last_hour": 6,
    "threshold": 5,
    "last_editor": "User123"
  }
}
```

### Usage Notes

- The system runs continuously until stopped
- Metrics are saved every 60 seconds
- All events are logged in real-time
- Database can be queried while system is running
- For production use, consider adding:
  - Error recovery and reconnection logic
  - More sophisticated alert rules
  - Email or webhook notifications
  - Dashboard for real-time monitoring